In [11]:
print("""
# Deep Learning Coding Flow
1. Imports
↓  
2. Dataset
↓  
3. Preprocessing
	- Image
	- Audio
↓
4. Model
	- Base Model
	- Implemented Model
↓
5. Optimizer & Loss
↓
6. Training Loop""")


# Deep Learning Coding Flow
1. Imports
↓  
2. Dataset
↓  
3. Preprocessing
	- Image
	- Audio
↓
4. Model
	- Base Model
	- Implemented Model
↓
5. Optimizer & Loss
↓
6. Training Loop


In [13]:
import os
import copy
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader


In [14]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Setup complete.")
print("Device:", device)

✅ Setup complete.
Device: cuda


In [15]:
# Image Transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [16]:
# Load Dataset
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("\nClasses: ", class_names)
print("\n Classes length",num_classes )


Classes:  ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted_plant', 'stop_sign', 'traffic_light', 'train', 'truck']

 Classes length 26


In [17]:
# Load Pretrained Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = models.vgg16(pretrained=True)

print(model)

cuda
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilat

In [18]:
for param in model.features.parameters():
    param.requires_grad = False

In [19]:
import torch.nn as nn

# 1. Modify the average pooling layer on your VGG model
model.avgpool = nn.AdaptiveAvgPool2d((1, 1))

# 2. Define a compact classifier head
model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.4),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.BatchNorm1d(128),
    nn.Dropout(0.3),
    nn.Linear(128, 26)
)

model = model.to(device)

In [20]:
# Loss and Optimizer
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


In [21]:
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_weights = None


In [22]:
BEST_WEIGHTS_PATH = 'models/vgg16_best.pth'
best_val_acc = 0.0  # initialize before training loop

for epoch in range(1, 15 + 1):

    # ----------------- Training phase -----------------
    model.train()
    model.features.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    train_loss = running_loss / total
    train_acc = running_correct / total

    # ----------------- Validation phase -----------------
    model.eval()
    val_running_loss = 0.0
    val_running_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            val_running_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += inputs.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_running_correct / val_total

    # ----------------- Scheduler step -----------------
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:02d}/{15} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e}"
    )

    # ----------------- Early stopping + save-best check -----------------
    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())

        # Save best weights to disk immediately, so we always have the
        # latest "best" version even if training is interrupted.
        torch.save(best_model_weights, BEST_WEIGHTS_PATH)
        print(f"  -> New best val_acc={val_acc:.4f}. Weights saved to '{BEST_WEIGHTS_PATH}'.")
    else:
        epochs_no_improve += 1
        EARLY_STOPPING_PATIENCE = 7
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\nNo improvement for {EARLY_STOPPING_PATIENCE} epochs "
                  f"-> stopping early at epoch {epoch}.")
            break

Epoch 01/15 | train_loss=2.2282 train_acc=0.3918 | val_loss=1.5152 val_acc=0.6077 | lr=1.00e-03
  -> New best val_acc=0.6077. Weights saved to 'models/vgg16_best.pth'.
Epoch 02/15 | train_loss=1.4116 train_acc=0.6077 | val_loss=1.2480 val_acc=0.6410 | lr=1.00e-03
  -> New best val_acc=0.6410. Weights saved to 'models/vgg16_best.pth'.
Epoch 03/15 | train_loss=1.1415 train_acc=0.6588 | val_loss=1.1392 val_acc=0.6590 | lr=1.00e-03
  -> New best val_acc=0.6590. Weights saved to 'models/vgg16_best.pth'.
Epoch 04/15 | train_loss=1.0307 train_acc=0.6984 | val_loss=1.0640 val_acc=0.6641 | lr=1.00e-03
  -> New best val_acc=0.6641. Weights saved to 'models/vgg16_best.pth'.
Epoch 05/15 | train_loss=0.9327 train_acc=0.7170 | val_loss=1.0396 val_acc=0.6846 | lr=1.00e-03
  -> New best val_acc=0.6846. Weights saved to 'models/vgg16_best.pth'.
Epoch 06/15 | train_loss=0.8624 train_acc=0.7319 | val_loss=1.0476 val_acc=0.6667 | lr=1.00e-03
Epoch 07/15 | train_loss=0.8761 train_acc=0.7115 | val_loss=1.02

In [23]:
print(f"Best model weights loaded. Best val_loss = {val_acc:.4f}")
print(f"Saved at: {BEST_WEIGHTS_PATH}")


Best model weights loaded. Best val_loss = 0.6974
Saved at: models/vgg16_best.pth
